<center>
  <img src="https://upload.wikimedia.org/wikipedia/commons/6/68/Logo_FPT_Education.png" width="300" alt="FPT Education logo" />
</center>

# III. SQL ANALYSIS

**Nguồn dữ liệu:** `SQL_Analysis/olist_analysis.db` - bảng `olist_analysis_dataset`

Bảng gồm 4 cột: `order_id`, `delay_days`, `item_count`, `low_review` (1 nếu `review_score <= 2`). Mỗi `order_id` chỉ xuất hiện một lần.

Mục tiêu: dùng SQL trả lời hai câu hỏi chính - giao trễ và số lượng sản phẩm trong đơn có làm tăng tỷ lệ đánh giá thấp không.

## 3.1. Kết nối SQLite

In [24]:
import sqlite3
from pathlib import Path

import pandas as pd

BASE_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "SQL_Analysis" / "olist_analysis.db").is_file())
DB_PATH = BASE_DIR / "SQL_Analysis" / "olist_analysis.db"
conn = sqlite3.connect(DB_PATH)

print(DB_PATH)

e:\ADY291c-AI2107-Group2\SQL_Analysis\olist_analysis.db


In [25]:
pd.read_sql_query("PRAGMA table_info(olist_analysis_dataset)", conn)[["name", "type"]]

,name,type
0,order_id,TEXT
1,delay_days,INTEGER
2,item_count,INTEGER
3,review_score,INTEGER
4,low_review,INTEGER


In [26]:
pd.read_sql_query("PRAGMA table_info(olist_analysis_dataset)", conn)[["name", "type"]]

,name,type
0,order_id,TEXT
1,delay_days,INTEGER
2,item_count,INTEGER
3,review_score,INTEGER
4,low_review,INTEGER


In [27]:
pd.read_sql_query("SELECT COUNT(*) AS total_orders FROM olist_analysis_dataset", conn)

,total_orders
0,95824


## 3.2. Phân tích giao hàng

**Câu hỏi:** Đơn giao trễ có tỷ lệ đánh giá thấp cao hơn đơn đúng hạn bao nhiêu?

Quy ước: `delay_days > 0` là **Late**, `delay_days <= 0` là **On time / Early**.

In [28]:
query = """
SELECT
    COUNT(*) AS total_orders,
    SUM(CASE WHEN delay_days > 0 THEN 1 ELSE 0 END) AS late_orders,
    SUM(CASE WHEN delay_days <= 0 THEN 1 ELSE 0 END) AS on_time_orders,
    SUM(low_review) AS low_review_orders
FROM olist_analysis_dataset
"""

pd.read_sql_query(query, conn)

,total_orders,late_orders,on_time_orders,low_review_orders
0,95824,6381,89443,12268


In [29]:
query = """
SELECT
    CASE
        WHEN delay_days > 0 THEN 'Late'
        ELSE 'On time / Early'
    END AS delivery_status,
    COUNT(*) AS total_orders,
    SUM(low_review) AS low_review_orders,
    ROUND(100.0 * SUM(low_review) / COUNT(*), 2) AS low_review_rate
FROM olist_analysis_dataset
GROUP BY delivery_status
ORDER BY delivery_status
"""

pd.read_sql_query(query, conn)

,delivery_status,total_orders,low_review_orders,low_review_rate
0,Late,6381,3980,62.37
1,On time / Early,89443,8288,9.27


**Nhận xét:** đơn giao trễ chiếm 6.381/95.824 đơn (6,66%) nhưng có tỷ lệ đánh giá thấp 62,37%, cao gấp gần 7 lần nhóm đúng hạn hoặc sớm (9,27%).

## 3.3. Phân tích số lượng sản phẩm trong đơn

**Câu hỏi:** Đơn có nhiều sản phẩm hơn có bị đánh giá thấp nhiều hơn không?

In [ ]:
query = """
SELECT
    item_count,
    COUNT(*) AS total_orders,
    SUM(low_review) AS low_review_orders,
    ROUND(100.0 * SUM(low_review) / COUNT(*), 2) AS low_review_rate
FROM olist_analysis_dataset
GROUP BY item_count
ORDER BY item_count
"""

pd.read_sql_query(query, conn)

,item_count,total_orders,low_review_orders,low_review_rate
0,1,86296,9711,11.25
1,2,7319,1864,25.47
2,3,1286,377,29.32
3,4,487,168,34.50
4,5,193,61,31.61
5,6,186,63,33.87
6,7,22,10,45.45
7,8,8,1,12.50
8,9,3,1,33.33
9,10,8,5,62.50


**Nhận xét:** tỷ lệ đánh giá thấp tăng từ 11,25% (1 sản phẩm) lên 25,47% (2 sản phẩm) và khoảng 30-35% khi đơn có từ 3 sản phẩm trở lên; các mức từ 8 sản phẩm trở lên chỉ có vài đơn nên tỷ lệ dao động mạnh.

## 3.4. Phân tích nhóm giao trễ

**Câu hỏi:** Tỷ lệ đánh giá thấp thay đổi thế nào khi số ngày trễ tăng dần?

Nhóm: `delay_days <= 0` (Early / On time), 1-3 ngày, 4-7 ngày, trên 7 ngày.

In [ ]:
query = """
SELECT
    CASE
        WHEN delay_days <= 0 THEN 'Early / On time'
        WHEN delay_days BETWEEN 1 AND 3 THEN '1-3 days late'
        WHEN delay_days BETWEEN 4 AND 7 THEN '4-7 days late'
        ELSE 'Over 7 days late'
    END AS delay_group,
    COUNT(*) AS total_orders,
    SUM(low_review) AS low_review_orders,
    ROUND(100.0 * SUM(low_review) / COUNT(*), 2) AS low_review_rate
FROM olist_analysis_dataset
GROUP BY delay_group
ORDER BY MIN(delay_days)
"""

pd.read_sql_query(query, conn)

,delay_group,total_orders,low_review_orders,low_review_rate
0,Early / On time,89443,8288,9.27
1,1-3 days late,1852,596,32.18
2,4-7 days late,1748,1182,67.62
3,Over 7 days late,2781,2202,79.18


**Nhận xét:** tỷ lệ đánh giá thấp tăng đều theo số ngày trễ: 9,27% -> 32,18% -> 67,62% -> 79,18%.

## 3.5. Phân tích kết hợp giao hàng và số lượng sản phẩm

**Câu hỏi:** Hai yếu tố giao trễ và số lượng sản phẩm kết hợp có làm tỷ lệ đánh giá thấp cao hơn không?

Nhóm sản phẩm: 1 item, 2-3 items, 4+ items.

In [ ]:
query = """
SELECT
    CASE
        WHEN delay_days > 0 THEN 'Late'
        ELSE 'On time / Early'
    END AS delivery_status,
    CASE
        WHEN item_count = 1 THEN '1 item'
        WHEN item_count BETWEEN 2 AND 3 THEN '2-3 items'
        ELSE '4+ items'
    END AS item_group,
    COUNT(*) AS total_orders,
    SUM(low_review) AS low_review_orders,
    ROUND(100.0 * SUM(low_review) / COUNT(*), 2) AS low_review_rate
FROM olist_analysis_dataset
GROUP BY delivery_status, item_group
ORDER BY delivery_status, item_group
"""

pd.read_sql_query(query, conn)

,delivery_status,item_group,total_orders,low_review_orders,low_review_rate
0,Late,1 item,5889,3654,62.05
1,Late,2-3 items,446,293,65.70
2,Late,4+ items,46,33,71.74
3,On time / Early,1 item,80407,6057,7.53
4,On time / Early,2-3 items,8159,1948,23.88
5,On time / Early,4+ items,877,283,32.27


**Nhận xét:** trong cùng nhóm giao hàng, đơn nhiều sản phẩm luôn có tỷ lệ đánh giá thấp cao hơn; mức cao nhất là Late + 4+ items (71,74%) và thấp nhất là On time + 1 item (7,53%).

## 3.6. Tổng kết

- Giao trễ là yếu tố ảnh hưởng mạnh nhất: 62,37% đơn trễ bị đánh giá thấp, so với 9,27% ở đơn đúng hạn.
- Mức độ trễ càng lớn thì tỷ lệ đánh giá thấp càng cao, đạt 79,18% khi trễ trên 7 ngày.
- Số lượng sản phẩm trong đơn cũng làm tăng tỷ lệ đánh giá thấp (11,25% với 1 sản phẩm lên khoảng 30% với 3 sản phẩm trở lên).
- Hai yếu tố kết hợp cho thấy nhóm rủi ro cao nhất là đơn giao trễ và có từ 4 sản phẩm trở lên.